# Final Project - Data 360

### Alivia Preston

##### International Union for Conservation of Nature (IUCN) has a red list of threatened species. This inventory includes the global conservation status of various animals, plants, and fungi. This project focuses on the endangered species from Utah (specifically, the first 294 species in alphabetical order)


In [1]:
import requests
from bs4 import BeautifulSoup
import re
import numpy as np
import pandas as pd

/Users/aliviapreston/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/aliviapreston/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Web scrapping

The needed information regarding the species on interest was scrapped from the web using `inspect` on the web page. This information was copied into a text file and used for this assignment to have static information

In [2]:
file_path = 'webscrapping.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    html_content = file.read()

In [3]:
soup = BeautifulSoup(html_content, 'html.parser')

In [4]:
species_blocks = soup.find_all('li', class_='list-results__item')

In [5]:
species = []
for block in species_blocks:
    classification = block.find('span', class_='list-results__classification')
    common_name = block.find('h2', class_='list-results__title')
    scientific_name = block.find('p', class_='list-results__subtitle')
    population = block.find('span', class_='species-population')
    assessment = block.find('span', class_='species-assessment')
    status_tag = block.find('a', class_='species-category')
    link = block.find_all('a')[-1]['href'] if block.find_all('a') else None

    species.append({
        'Classification': classification.text.strip() if classification else '',
        'Common Name': common_name.text.strip() if common_name else '',
        'Scientific Name': scientific_name.text.strip() if scientific_name else '',
        'Population Trend': population.text.strip() if population else '',
        'Assessment': assessment.text.strip() if assessment else '',
        'Conservation Status': status_tag['title'] if status_tag and 'title' in status_tag.attrs else '',
        'IUCN Link': f"https://www.iucnredlist.org{link}" if link else ''
    })
#species

In [6]:
df = pd.DataFrame(species)
df

,Classification,Common Name,Scientific Name,Population Trend,Assessment,Conservation Status,IUCN Link
0,plantae — polypodiopsida,Maidenhair Fern,Adiantum capillus-veneris,Stable,Global,Least Concern,https://www.iucnredlist.org/species/164082/677...
1,animalia — insecta,Lance-tipped Darner,Aeshna constricta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959122/6...
2,animalia — insecta,Lake Darner,Aeshna eremita,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959126/6...
3,animalia — insecta,Variable Darner,Aeshna interrupta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959132/6...
4,animalia — insecta,Moorland Hawker,Aeshna juncea,Stable,Global,Least Concern,https://www.iucnredlist.org/species/165518/658...
...,...,...,...,...,...,...,...
289,plantae — magnoliopsida,,Fraxinus velutina,Unknown,Global,Least Concern,https://www.iucnredlist.org/species/96444728/9...
290,plantae — liliopsida,,Galanthus nivalis,Decreasing,"Global, Europe",Near Threatened,https://www.iucnredlist.org/species/162168/555...
291,animalia — gastropoda,Prairie Fossaria,Galba bulimoides,Stable,Global,Least Concern,https://www.iucnredlist.org/species/189207/870...
292,animalia — gastropoda,Rock Fossaria,Galba modicella,Stable,Global,Least Concern,https://www.iucnredlist.org/species/69620662/6...


#### Split the classification into kingdom and class

In [7]:
df[['Kingdom', 'Class']] = df['Classification'].str.split('—', expand=True)
df['Kingdom'] = df['Kingdom'].str.strip()
df['Class'] = df['Class'].str.strip()
df

,Classification,Common Name,Scientific Name,Population Trend,Assessment,Conservation Status,IUCN Link,Kingdom,Class
0,plantae — polypodiopsida,Maidenhair Fern,Adiantum capillus-veneris,Stable,Global,Least Concern,https://www.iucnredlist.org/species/164082/677...,plantae,polypodiopsida
1,animalia — insecta,Lance-tipped Darner,Aeshna constricta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959122/6...,animalia,insecta
2,animalia — insecta,Lake Darner,Aeshna eremita,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959126/6...,animalia,insecta
3,animalia — insecta,Variable Darner,Aeshna interrupta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959132/6...,animalia,insecta
4,animalia — insecta,Moorland Hawker,Aeshna juncea,Stable,Global,Least Concern,https://www.iucnredlist.org/species/165518/658...,animalia,insecta
...,...,...,...,...,...,...,...,...,...
289,plantae — magnoliopsida,,Fraxinus velutina,Unknown,Global,Least Concern,https://www.iucnredlist.org/species/96444728/9...,plantae,magnoliopsida
290,plantae — liliopsida,,Galanthus nivalis,Decreasing,"Global, Europe",Near Threatened,https://www.iucnredlist.org/species/162168/555...,plantae,liliopsida
291,animalia — gastropoda,Prairie Fossaria,Galba bulimoides,Stable,Global,Least Concern,https://www.iucnredlist.org/species/189207/870...,animalia,gastropoda
292,animalia — gastropoda,Rock Fossaria,Galba modicella,Stable,Global,Least Concern,https://www.iucnredlist.org/species/69620662/6...,animalia,gastropoda


In [8]:
df.to_csv('iucn_species_scraped.csv', index=False)
print(df.head())

             Classification          Common Name            Scientific Name  \
0  plantae — polypodiopsida      Maidenhair Fern  Adiantum capillus-veneris   
1        animalia — insecta  Lance-tipped Darner          Aeshna constricta   
2        animalia — insecta          Lake Darner             Aeshna eremita   
3        animalia — insecta      Variable Darner          Aeshna interrupta   
4        animalia — insecta      Moorland Hawker              Aeshna juncea   

  Population Trend Assessment Conservation Status  \
0           Stable     Global       Least Concern   
1           Stable     Global       Least Concern   
2           Stable     Global       Least Concern   
3           Stable     Global       Least Concern   
4           Stable     Global       Least Concern   

                                           IUCN Link   Kingdom           Class  
0  https://www.iucnredlist.org/species/164082/677...   plantae  polypodiopsida  
1  https://www.iucnredlist.org/species/50959

# part 2 with the entire list 

In [9]:
file_path = 'webscrape2.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    html_content = file.read()

In [10]:
soup2 = BeautifulSoup(html_content, 'html.parser')

In [11]:
species_blocks2 = soup2.find_all('li', class_='list-results__item')

In [12]:
species2 = []
for block in species_blocks2:
    classification = block.find('span', class_='list-results__classification')
    common_name = block.find('h2', class_='list-results__title')
    scientific_name = block.find('p', class_='list-results__subtitle')
    population = block.find('span', class_='species-population')
    assessment = block.find('span', class_='species-assessment')
    status_tag = block.find('a', class_='species-category')
    link = block.find_all('a')[-1]['href'] if block.find_all('a') else None

    species2.append({
        'Classification': classification.text.strip() if classification else '',
        'Common Name': common_name.text.strip() if common_name else '',
        'Scientific Name': scientific_name.text.strip() if scientific_name else '',
        'Population Trend': population.text.strip() if population else '',
        'Assessment': assessment.text.strip() if assessment else '',
        'Conservation Status': status_tag['title'] if status_tag and 'title' in status_tag.attrs else '',
        'IUCN Link': f"https://www.iucnredlist.org{link}" if link else ''
    })
#species

In [13]:
df = pd.DataFrame(species2)
df

,Classification,Common Name,Scientific Name,Population Trend,Assessment,Conservation Status,IUCN Link
0,plantae — polypodiopsida,Maidenhair Fern,Adiantum capillus-veneris,Stable,Global,Least Concern,https://www.iucnredlist.org/species/164082/677...
1,animalia — insecta,Lance-tipped Darner,Aeshna constricta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959122/6...
2,animalia — insecta,Lake Darner,Aeshna eremita,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959126/6...
3,animalia — insecta,Variable Darner,Aeshna interrupta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959132/6...
4,animalia — insecta,Moorland Hawker,Aeshna juncea,Stable,Global,Least Concern,https://www.iucnredlist.org/species/165518/658...
...,...,...,...,...,...,...,...
781,plantae — liliopsida,,Yucca toftiae,Decreasing,Global,Vulnerable,https://www.iucnredlist.org/species/172050021/...
782,plantae — liliopsida,,Yucca utahensis,Decreasing,Global,Least Concern,https://www.iucnredlist.org/species/117428761/...
783,plantae — magnoliopsida,,Zapoteca formosa,Stable,Global,Least Concern,https://www.iucnredlist.org/species/19891812/2...
784,animalia — mammalia,,Zapus princeps,Stable,Global,Least Concern,https://www.iucnredlist.org/species/42614/1151...


In [14]:
df[['Kingdom', 'Class']] = df['Classification'].str.split('—', expand=True)
df['Kingdom'] = df['Kingdom'].str.strip()
df['Class'] = df['Class'].str.strip()
df

,Classification,Common Name,Scientific Name,Population Trend,Assessment,Conservation Status,IUCN Link,Kingdom,Class
0,plantae — polypodiopsida,Maidenhair Fern,Adiantum capillus-veneris,Stable,Global,Least Concern,https://www.iucnredlist.org/species/164082/677...,plantae,polypodiopsida
1,animalia — insecta,Lance-tipped Darner,Aeshna constricta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959122/6...,animalia,insecta
2,animalia — insecta,Lake Darner,Aeshna eremita,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959126/6...,animalia,insecta
3,animalia — insecta,Variable Darner,Aeshna interrupta,Stable,Global,Least Concern,https://www.iucnredlist.org/species/50959132/6...,animalia,insecta
4,animalia — insecta,Moorland Hawker,Aeshna juncea,Stable,Global,Least Concern,https://www.iucnredlist.org/species/165518/658...,animalia,insecta
...,...,...,...,...,...,...,...,...,...
781,plantae — liliopsida,,Yucca toftiae,Decreasing,Global,Vulnerable,https://www.iucnredlist.org/species/172050021/...,plantae,liliopsida
782,plantae — liliopsida,,Yucca utahensis,Decreasing,Global,Least Concern,https://www.iucnredlist.org/species/117428761/...,plantae,liliopsida
783,plantae — magnoliopsida,,Zapoteca formosa,Stable,Global,Least Concern,https://www.iucnredlist.org/species/19891812/2...,plantae,magnoliopsida
784,animalia — mammalia,,Zapus princeps,Stable,Global,Least Concern,https://www.iucnredlist.org/species/42614/1151...,animalia,mammalia


In [15]:
df.to_csv('iucn_species_scraped2.csv', index=False)
print(df.head())

             Classification          Common Name            Scientific Name  \
0  plantae — polypodiopsida      Maidenhair Fern  Adiantum capillus-veneris   
1        animalia — insecta  Lance-tipped Darner          Aeshna constricta   
2        animalia — insecta          Lake Darner             Aeshna eremita   
3        animalia — insecta      Variable Darner          Aeshna interrupta   
4        animalia — insecta      Moorland Hawker              Aeshna juncea   

  Population Trend Assessment Conservation Status  \
0           Stable     Global       Least Concern   
1           Stable     Global       Least Concern   
2           Stable     Global       Least Concern   
3           Stable     Global       Least Concern   
4           Stable     Global       Least Concern   

                                           IUCN Link   Kingdom           Class  
0  https://www.iucnredlist.org/species/164082/677...   plantae  polypodiopsida  
1  https://www.iucnredlist.org/species/50959

## Streamlit App Building

The streamlit app was created in VScode and uploaded to GitHub